# Entrainement YOLO

Objectif:

- charger un dataset ZIP
- entrainer un modele YOLO
- evaluer le modele
- recuperer `best.pt`

## 1. Initialisation

In [ ]:
!nvidia-smi
!pip install -q ultralytics pyyaml

In [ ]:
from pathlib import Path
import shutil
import zipfile
import yaml
import torch
from ultralytics import YOLO

WORKDIR = Path('/content/wire_harness_training')
WORKDIR.mkdir(parents=True, exist_ok=True)

print('Torch :', torch.__version__)
print('GPU   :', torch.cuda.is_available())
print('Dossier de travail :', WORKDIR)

## 2. Charger le dataset ZIP

L'archive doit contenir:

```text
data.yaml
images/train
images/val
labels/train
labels/val
```

In [ ]:
from google.colab import files

uploaded = files.upload()
zip_name = next(iter(uploaded))
zip_path = Path(zip_name)

extract_dir = WORKDIR / 'dataset'
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)

yaml_candidates = list(extract_dir.rglob('data.yaml'))
assert yaml_candidates, 'Aucun fichier data.yaml trouve dans l archive.'
dataset_yaml = yaml_candidates[0]
dataset_dir = dataset_yaml.parent

print('Archive chargee :', zip_path)
print('Dataset extrait :', dataset_dir)
print('YAML retenu     :', dataset_yaml)

In [ ]:
with open(dataset_yaml, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f) or {}

runtime_yaml = dataset_dir / 'data_runtime.yaml'
data_cfg['path'] = str(dataset_dir.resolve())

with open(runtime_yaml, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False, allow_unicode=True)

dataset_yaml = runtime_yaml

print('YAML corrige :', dataset_yaml)
print(data_cfg)


In [ ]:
train_images = list((dataset_dir / 'images' / 'train').glob('*'))
val_images = list((dataset_dir / 'images' / 'val').glob('*'))
train_labels = list((dataset_dir / 'labels' / 'train').glob('*.txt'))
val_labels = list((dataset_dir / 'labels' / 'val').glob('*.txt'))

print('Images train :', len(train_images))
print('Images val   :', len(val_images))
print('Labels train :', len(train_labels))
print('Labels val   :', len(val_labels))

## 3. Parametres d entrainement

In [ ]:
PROJECT_NAME = 'wire_harness_yolo_real'
BASE_MODEL = 'yolov8n.pt'
EPOCHS = 50
IMGSZ = 640
BATCH = 16
TEST_SOURCE = ''

print('Projet      :', PROJECT_NAME)
print('Base model  :', BASE_MODEL)
print('Epochs      :', EPOCHS)
print('Image size  :', IMGSZ)
print('Batch       :', BATCH)

## 4. Entrainement

In [ ]:
device = 0 if torch.cuda.is_available() else 'cpu'
model = YOLO(BASE_MODEL)

train_results = model.train(
    data=str(dataset_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=device,
    project=str(WORKDIR / 'runs'),
    name=PROJECT_NAME,
    exist_ok=True,
    pretrained=True,
    workers=2
)

print('Entrainement termine.')

## 5. Recuperation du modele

In [ ]:
best_model_path = WORKDIR / 'runs' / PROJECT_NAME / 'weights' / 'best.pt'
last_model_path = WORKDIR / 'runs' / PROJECT_NAME / 'weights' / 'last.pt'

print('best.pt :', best_model_path)
print('last.pt :', last_model_path)

assert best_model_path.exists(), 'Le fichier best.pt est introuvable.'

## 6. Evaluation

In [ ]:
best_model = YOLO(str(best_model_path))
val_results = best_model.val(data=str(dataset_yaml), device=device)
print(val_results)

## 7. Test d inference

In [ ]:
if TEST_SOURCE:
    pred_results = best_model.predict(
        source=TEST_SOURCE,
        conf=0.25,
        save=True,
        project=str(WORKDIR / 'predictions'),
        name='test',
        exist_ok=True,
        device=device
    )
    print('Resultats :', WORKDIR / 'predictions' / 'test')
else:
    print('TEST_SOURCE vide : aucun test supplementaire lance.')

## 8. Telechargement

In [ ]:
from google.colab import files
files.download(str(best_model_path))

## 9. Integration locale

Apres telechargement:

1. copier `best.pt` dans `data/models/`
2. garder dans `config/app.yaml`:

```yaml
detector:
  mode: yolo
  yolo_model_path: data/models/best.pt
```

3. lancer localement:

```bash
python run.py
```